## Tracking of a Mosquito-path dataset optimised over a parameter grid, using likelihoods

In [ ]:
## Non-optimised params
num_timesteps= 450
number_particles = 200
seed = 1 # Random seem for reproducibility
c=10

In [ ]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import pandas as pd
from stonesoup.types.detection import Detection
from stonesoup.types.groundtruth import GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import CovarianceMatrices, StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater

# Step 1: Load the CSV file- With thanks to Chloe Chung for the data!
excel_folder = fr"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TrackedDatasets\mosquito data"
files = [
    rf"fly_1",
    rf"fly_2",
    rf"mosquito_1",
    rf"mosquito_2",
    rf"mosquito_3_with_velocity",
    rf"mosquito_4_with_velocity"
]

start_time=datetime.now()

# For each object, collect identical tracks of detection objects for updating, and groundtruths for plotting.
trajectory_data={'dimensions':{},'measurements':{},'plottable_tracks':{}, 'prior_stats':{}, 'prior_states':{}}

for item in files:
    file_path = excel_folder + "\\" + item + ".csv"
    data = pd.read_csv(file_path)
    columns = data.columns

    has_vel = ('vel_x' in columns and 'vel_y' in columns)
    has_z   = ('z' in columns)
    trajectory_data['dimensions'][item] = (has_vel,has_z)
    # Extract x,y (and z, vel_x, vel_y if present)
    if num_timesteps>len(data['x']):
        num_timesteps=len(data['x'])
    x_arr = data['x'][:num_timesteps]
    y_arr = data['y'][:num_timesteps]
    z_arr = data['z'][:num_timesteps] if has_z else None
    
    velx_arr = data['vel_x'][:num_timesteps] if has_vel else None
    vely_arr = data['vel_y'][:num_timesteps] if has_vel else None
    
    timesteps = [start_time + timedelta(seconds=i) for i in range(len(data))]
    timesteps = timesteps[:num_timesteps]

    # Build separate Tracks of Detections for position vs velocity
    measurement_track = Track()
    plottable_track= Track()
    for i in range(num_timesteps):
        timestamp = timesteps[i]
        if has_z:
            # 3D position [x_i, y_i, z_i]
            meas = np.array([x_arr[i], y_arr[i], z_arr[i]])
            state = meas
            if i==0:
                prior_mean =state

        elif has_vel:
            # 2Dx2 position [x_i, x_vel_i, y_i, y_vel_i]
            meas = np.array([x_arr[i], y_arr[i]])
            state = np.array([x_arr[i], velx_arr[i], y_arr[i], vely_arr[i]])
            if i==0:
                prior_mean = state

        detection = Detection(
            state_vector=StateVector(meas),
            timestamp=timestamp,
            metadata={"object": item}
        )        
        plottable_state = GroundTruthState(
            state_vector=StateVector(state),
            timestamp=timestamp,
            metadata={"object": item})
        measurement_track.append(detection)
        plottable_track.append(plottable_state)
    x_std = np.std(x_arr)
    y_std = np.std(y_arr)
    if has_vel:
        vx_std = np.std(velx_arr)
        vy_std = np.std(vely_arr)
        prior_covar = np.diag([x_std**2, vx_std**2, y_std**2, vy_std**2])
    elif has_z:
        z_std = np.std(z_arr)
        prior_covar = np.diag([x_std**2, y_std**2, z_std**2])
    trajectory_data['measurements'][item] = measurement_track
    trajectory_data['plottable_tracks'][item]=plottable_track
    trajectory_data['prior_stats'][item]=(prior_mean,prior_covar) #first store mean and covar in dict

In [ ]:
#Determine assumed measurement std.
meas_sigma=0.00005
meas_sig2 = meas_sigma**2

In [ ]:
## Initiate Priors
for item in files:
    prior_mean,prior_covar=trajectory_data['prior_stats'][item]
    # Sample from the prior Gaussian distribution
    states = multivariate_normal.rvs(
        mean=prior_mean,
        cov=prior_covar,  # Covariance for the initial state
        size=number_particles
    )

    # Define covariance for particle array
    covars = [prior_covar for _ in range(number_particles)]

    # Create prior particle state
    lp_prior = MarginalisedParticleState(
        state_vector=StateVectors(states.T),  # Transpose states to shape (2, N)
        covariance=CovarianceMatrices(covars).T,  # Covariance matrix
        weight=np.array([Probability(1 / number_particles)] * number_particles),
        timestamp=start_time-timedelta(milliseconds=1)
    )
    gp_prior=GaussianState(state_vector=prior_mean,
                            covar=prior_covar,
                            timestamp=start_time-timedelta(milliseconds=1))
    # Store them in the dictionary
    trajectory_data['prior_states'][item]=(lp_prior,gp_prior) # lp and gp prior objects

In [ ]:
## Outlining the parameter grids over which we'll estimate
# RandomWalk parameters    
#  
#q or sigma_W, std of noise
noise_diff_coeffs = np.logspace(-15, -10, num= 3)
#alpha param, how heavy-tailed levy distr is. avoid alpha=1      
a,b=2,2
pre_1_alpha,post_1_alpha, alpha_values= np.linspace(0.5, 0.9, a), np.linspace(1.1,1.9,b), np.zeros(a+b)
alpha_values[:a],alpha_values[a:a+b]=pre_1_alpha,post_1_alpha
# See below, mu_w=0 held const.

#Levy Langevin or Ulhenbeck Orstein params:
damping_coeffs= np.linspace(0.01, 0.5, 3)

In [ ]:
## Choose which objects to track
generate_models_for_items=[ #comment out items to avoid wasting time generating their models. 
    rf"fly_1",
    rf"fly_2",
    rf"mosquito_1",
    rf"mosquito_2",
    rf"mosquito_3_with_velocity",
    rf"mosquito_4_with_velocity"
]

In [ ]:
## Generates the necessary predictor dicts for all the components to save time in the loop
from stonesoup.models.transition.levy_linear import LevyRandomWalk
from stonesoup.models.transition.linear import CombinedLinearGaussianTransitionModel, OrnsteinUhlenbeck

resampler = SystematicResampler()

lp_predictors= {}
gp_predictors = {}

for item in generate_models_for_items:
    has_vel,has_z= trajectory_data['dimensions'][item]
    lp_predictors[item]= {}
    gp_predictors[item]= {}
    for q in noise_diff_coeffs: 
        for theta in damping_coeffs: # how quickly decay back to zero occurs (higher=quicker)
            for alpha in alpha_values: # how heavy-tailed distr is (lower=heavier)
                    
                    #Generates all the Levy process predictors and updaters
                    #driver has sigma_W=1, so std. controlled by noise_diff_coeff input into model
                    lp_driver = AlphaStableNSMDriver(mu_W=0, sigma_W2=q**2, seed=seed, c=c, alpha=alpha, noise_case=NoiseCase(2))                            
                    if has_vel:
                        langevin_x = LevyLangevin(driver=lp_driver, noise_diff_coeff=q, damping_coeff=theta)
                        langevin_y=langevin_x
                        lp_transition_model=CombinedLinearLevyTransitionModel([langevin_x,langevin_y])
                    elif has_z:
                        levy_rw_x=LevyRandomWalk(driver=lp_driver,noise_diff_coeff=q)
                        levy_rw_y=levy_rw_x
                        levy_rw_z=levy_rw_x
                        lp_transition_model=CombinedLinearLevyTransitionModel([levy_rw_x,levy_rw_y,levy_rw_z])
                    lp_predictor = MarginalisedParticlePredictor(transition_model=lp_transition_model)
                    lp_predictors[item][(q,theta, alpha)] = lp_predictor

            #Generates all the Gaussian process predictors and updaters
            if has_vel:
                gp_OU_x=OrnsteinUhlenbeck(noise_diff_coeff=q, damping_coeff=theta)
                gp_OU_y=gp_OU_x
                gp_transition_model=CombinedLinearGaussianTransitionModel([gp_OU_x,gp_OU_y])
            elif has_z:
                gp_RW_x= RandomWalk(noise_diff_coeff=q)
                gp_RW_y=gp_RW_x
                gp_RW_z=gp_RW_x
                gp_transition_model= CombinedLinearGaussianTransitionModel([gp_RW_x,gp_RW_y,gp_RW_z])
            gp_predictor = KalmanPredictor(gp_transition_model)
            gp_predictors[item][(q,theta)] = gp_predictor     

In [ ]:
## Build the updaters that depend on measurement model only
from stonesoup.updater.tests.conftest import measurement_model

lp_updaters={}
gp_updaters={}
for item in generate_models_for_items:
    has_vel,has_z= trajectory_data['dimensions'][item]
    if has_vel:
        ndim_state=4
        mapping=[0,2]
    elif has_z:
        ndim_state=3
        mapping=[0,1,2]
    measurement_model = LinearGaussian(
    ndim_state=ndim_state,  # State vector dimensions: [price, dP/dt] for the models
    mapping=mapping,  # Map the measurement to the 'price' dimension
    noise_covar=np.diag([meas_sig2]*len(mapping)))

    lp_updaters[item] = MarginalisedParticleUpdater(measurement_model, resampler)
    gp_updaters[item] = KalmanUpdater(measurement_model)   

In [ ]:
## Filtering and likelihood calculation
from scipy.special import logsumexp
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

#Track dicts
lp_tracks = {}
gp_tracks = {}

#Likelihood dicts
lp_likelihoods = {}
gp_likelihoods = {}

measurement_dict=trajectory_data['measurements']

for j, item in enumerate(generate_models_for_items):
    measurements=measurement_dict[item]
    lp_updater=lp_updaters[item]
    gp_updater=gp_updaters[item]
    lp_likelihoods[item] = {}
    gp_likelihoods[item] = {}
    lp_tracks[item]={}
    gp_tracks[item]={}

    for i, meas in enumerate(measurements):
        # Possibly you'd also keep track of prior states for each param combination
        # For each alpha, etc.:
        for q in noise_diff_coeffs:
            for theta in damping_coeffs:
                for alpha in alpha_values:
                    if (q,theta,alpha) not in lp_likelihoods[item]:
                        lp_tracks[item][(q,theta, alpha)]=Track()
                        prior=trajectory_data['prior_states'][item][0]
                        lp_likelihoods[item][(q,theta, alpha)] = 0.0
                    else:
                        lp_tracks[item][(q,theta, alpha)]
                        prior = lp_tracks[item][(q,theta, alpha)][-1]
                        
                    lp_predictor = lp_predictors[item][(q,theta, alpha)]
                    lp_prediction = lp_predictor.predict(prior, timestamp=meas.timestamp)
                    lp_hypothesis = SingleHypothesis(lp_prediction, meas)                    
                    lp_post = lp_updater.update(lp_hypothesis)
                    lp_tracks[item][(q,theta, alpha)].append(lp_post)
                    # Accumulate log-likelihood
                    lp_likelihoods[item][(q,theta, alpha)] += logsumexp(lp_updater.measurement_model.logpdf(meas,lp_post))-np.log(number_particles*num_timesteps)

                if (q,theta) not in gp_likelihoods[item]:
                    gp_tracks[item][(q,theta)]=Track()
                    prior=trajectory_data['prior_states'][item][1]
                    gp_likelihoods[item][(q,theta)] = 0.0
                else:
                    gp_tracks[item][(q,theta)]
                    prior = gp_tracks[item][(q,theta)][-1]

                gp_predictor = gp_predictors[item][(q,theta)]
                gp_prediction = gp_predictor.predict(prior, timestamp=meas.timestamp)
                gp_hypothesis = SingleHypothesis(gp_prediction, meas)
                gp_post = gp_updater.update(gp_hypothesis)
                gp_tracks[item][(q,theta)].append(gp_post)
                
                gp_likelihoods[item][(q,theta)] += gp_updater.measurement_model.logpdf(meas,gp_post) -np.log(num_timesteps)

        print(f"object {j+1}/{len(generate_models_for_items)+1}, ({i+1}/{len(measurements)} measurements)")

In [ ]:
## Define likelihood organiser and output optimal params
def summarize_top_likelihoods(item,lp_likelihoods,gp_likelihoods, top_n=5):
    """
    Summarize and print the top-N parameter configurations with the highest 
    log-likelihood for both the Lévy process and Gaussian process models.
    
    Parameters
    ----------
    lp_likelihoods : dict
        Dictionary keyed by (mu_W, theta, alpha, q, meas_sig2),
        with values = total (log) likelihood.
    gp_likelihoods : dict
        Dictionary keyed by ((q,theta), meas_sig2), with values = total (log) likelihood.
    top_n : int, optional
        Number of highest-likelihood entries to show for each model. Default=5.
    
    Returns
    -------
    list_of_top_lp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Lévy model.
    list_of_top_gp : list
        Sorted list of tuples [(params, log_like), ...] from high to low log-like 
        for the Gaussian process model.
    """
    item_lp_likelihoods, item_gp_likelihoods = lp_likelihoods[item], gp_likelihoods[item]
    # --- 1) Sort the Lévy model likelihoods ---
    #   lp_likelihoods is keyed by (mu_W, theta, alpha, q, meas_sig2)
    #   The value is the total log-likelihood
    # We'll get items as ((q,theta, alpha), loglike)
    list_of_lp = list(item_lp_likelihoods.items())
    # Sort descending by log-likelihood
    list_of_lp.sort(key=lambda x: x[1], reverse=True)
    # Take top_n
    list_of_top_lp = list_of_lp[:top_n]

    # --- 2) Sort the Gaussian process likelihoods ---
    #   gp_likelihoods is keyed by (q,theta)
    list_of_gp = list(item_gp_likelihoods.items())
    list_of_gp.sort(key=lambda x: x[1], reverse=True)
    list_of_top_gp = list_of_gp[:top_n]

    # --- 3) Print summary in a neat format ---
    print(f"=== Lévy Langevin Process -{item} -Top {top_n} Log-Likelihoods ===")
    for rank, ((q,theta, alpha), loglike) in enumerate(list_of_top_lp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | q={q}, alpha={alpha}") #, theta={theta} , mu={mu_W}")

    print("")
    print(f"=== Gaussian Ornstein_Uhlenbeck -{item} -Top {top_n} Log-Likelihoods ===")
    for rank, ((q,theta), loglike) in enumerate(list_of_top_gp, start=1):
        print(f"{rank:2d}) logL={loglike:.2f} | q={q}") #, theta={theta}")

    return list_of_top_lp, list_of_top_gp

In [ ]:
## Path to save plots in
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSDplots"

In [ ]:
## Optimal parameters used for tracks and plotting
lp_optimal_params ={}
gp_optimal_params ={}
Optimal_lp_tracks= {}
Optimal_gp_tracks={}
for item in generate_models_for_items:
    list_of_top_lp, list_of_top_gp = summarize_top_likelihoods(item,lp_likelihoods,gp_likelihoods, top_n=5)
    lp_optimal_params[item] = list_of_top_lp[0][0] 
    gp_optimal_params[item] = list_of_top_gp[0][0]
    Optimal_lp_tracks[item] = lp_tracks[item][lp_optimal_params[item]]
    Optimal_gp_tracks[item] = gp_tracks[item][gp_optimal_params[item]]

In [ ]:
## Whether to calculate the smoothed trajectories and how to plot them
plot_smooth=False
uncertainty=True
particle=False
plot_particle_paths=False
i=0
particle_plotter_dict = {}

In [ ]:
## Plotting
file_path = Path(folder_path + rf"\TrackingPlot.html")
file_path.parent.mkdir(parents=True, exist_ok=True)
for item in generate_models_for_items:
    particle_plotter_dict[item]= Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.ONE, axis_labels=[item])
    if i==0:
        particle_plotter_dict[item].plot_ground_truths(trajectory_data['plottable_tracks'][item], [i], truths_label="Observations")
    particle_plotter_dict[item].plot_tracks(Optimal_lp_tracks[item], [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Levy",line=dict(width=1))
    particle_plotter_dict[item].plot_tracks(Optimal_gp_tracks[item], [i],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Gaussian",line=dict(width=1))
    particle_plotter_dict[item].fig.update_layout( 
        plot_bgcolor="white",  # Set background color to white
        xaxis=dict(
            showgrid=True,
            gridcolor="gray",      # Keep the grid
            title=dict(text="Time", font=dict(size=20)),  # Add large item
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor="gray",      # Keep the grid
            title=dict(text=item, font=dict(size=20)),  # Add large item
        ),
        legend=dict(
            font=dict(size=15),       # Make the legend font larger
            # orientation='v',
            # xanchor="auto",         # Center the legend
            # yanchor="auto",           # Align the legend to the bottom of the plot
            bordercolor="Black",
            borderwidth=3,
            # y=+0.45,                   # Position it above the graph
            # x=0.6                    # Center it horizontally
        ),
    )
    particle_plotter_dict[item].fig.write_html(str(file_path))
    particle_plotter_dict[item].fig.show()

In [ ]:
particle_plotter_dict_2D_3D ={}
## Plotting
file_path = Path(folder_path + rf"\TrackingPlot.html")
file_path.parent.mkdir(parents=True, exist_ok=True)
particle=False
uncertainty=False
plot_particle_paths=False
for item in generate_models_for_items:
    if trajectory_data['dimensions'][item][1]:

        particle_plotter_dict_2D_3D[item]= Plotterly(autosize=False, width=1500,height=800, dimension=Dimension.THREE)
        if i==0:
            particle_plotter_dict_2D_3D[item].plot_ground_truths(trajectory_data['plottable_tracks'][item], [0,1,2], truths_label="Observations")
        particle_plotter_dict_2D_3D[item].plot_tracks(Optimal_lp_tracks[item], [0,1,2],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Levy",line=dict(width=1))
        particle_plotter_dict_2D_3D[item].plot_tracks(Optimal_gp_tracks[item], [0,1,2],mode="lines",uncertainty=uncertainty,particle=particle,plot_particle_paths=plot_particle_paths, track_label="Gaussian",line=dict(width=1))
        particle_plotter_dict_2D_3D[item].fig.update_layout( 
            plot_bgcolor="white",  # Set background color to white
            xaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text="Time", font=dict(size=20)),  # Add large item
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor="gray",      # Keep the grid
                title=dict(text=item, font=dict(size=20)),  # Add large item
            ),
            legend=dict(
                font=dict(size=15),       # Make the legend font larger
                # orientation='v',
                # xanchor="auto",         # Center the legend
                # yanchor="auto",           # Align the legend to the bottom of the plot
                bordercolor="Black",
                borderwidth=3,
                # y=+0.45,                   # Position it above the graph
                # x=0.6                    # Center it horizontally
            ),
        )
        particle_plotter_dict_2D_3D[item].fig.write_html(str(file_path))
        particle_plotter_dict_2D_3D[item].fig.show()

: 